# Recording Transcriber

Turns a meeting recording into a transcript with named speakers, a protocol and an agenda.

**Every time:** choose **Runtime → Run all**. To pick a specific job, type its folder name into `JOB` first (empty: the newest folder without a transcript).
After a disconnect, choose **Run all** again: the run continues where it stopped. Do not save changes to this notebook. Help: `how-to.md` next to this notebook.

In [ ]:
# @title 1. Connect Google Drive
JOB = ""  # @param {type:"string"}
WORKSPACE = "Transcriptions"  # @param {type:"string"}

from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
MY_DRIVE = Path("/content/drive/MyDrive")
# the app folder on the shared drive holds config.yaml, the wheel and this notebook
APP_FOLDERS = sorted(Path("/content/drive/Shareddrives").glob("*/audio_transcription"))
APP_FOLDERS = [folder for folder in APP_FOLDERS if (folder / "config.yaml").is_file()]
if not APP_FOLDERS:
    raise RuntimeError(
        "The app folder audio_transcription was not found on a shared drive. "
        "Ask your administrator to check that you can see it."
    )
APP = APP_FOLDERS[0]
if len(APP_FOLDERS) > 1:
    print(f"Several app folders found; using {APP}.")
print(f"App folder: {APP}")

In [ ]:
# @title 2. Install the transcriber
import importlib
import re
import subprocess
import sys
from importlib.metadata import version

import yaml

try:
    from packaging.version import InvalidVersion, Version
except ImportError:
    # pip always ships its own copy
    from pip._vendor.packaging.version import InvalidVersion, Version

PACKAGE = "recording-transcriber"
WHEEL_NAME = re.compile(r"recording_transcriber-([^-]+)-py3-none-any\.whl")


def pinned_version(settings):
    """Return app_version from config.yaml, or None; stop if it is not a quoted version."""
    pinned = settings.get("app_version")
    if pinned in (None, ""):
        return None
    if not isinstance(pinned, str):
        raise TypeError(
            'Quote app_version in config.yaml (e.g. "1.0.0"). Ask your administrator.'
        )
    try:
        Version(pinned)
    except InvalidVersion:
        raise RuntimeError(
            f"app_version {pinned!r} in config.yaml is not a version. Ask your administrator."
        ) from None
    return pinned


def pick_wheel(app, pinned=None):
    """Return the app-folder wheel to install: the pinned version, else the highest (release candidates count)."""
    wheels = {}
    for path in app.glob("recording_transcriber-*-py3-none-any.whl"):
        match = WHEEL_NAME.fullmatch(path.name)
        try:
            wheels[Version(match.group(1))] = path
        except (AttributeError, InvalidVersion):
            continue
    if pinned:
        return wheels.get(Version(pinned))
    return wheels[max(wheels)] if wheels else None


def install(target, source):
    """Install a wheel file or a PyPI requirement; stop with a message if pip fails."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", str(target)],
        capture_output=True,
        check=False,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError(
            f"Installing the transcriber from {source} failed. "
            "Run all again; if it fails again, ask your administrator."
        )
    # the package is imported in this process right after the install
    importlib.invalidate_caches()


if "recording_transcriber" in sys.modules:
    # a new version only takes effect in a fresh session (Runtime > Disconnect and delete runtime)
    print("The transcriber is already loaded in this session.")
else:
    try:
        SETTINGS = yaml.safe_load((APP / "config.yaml").read_text(encoding="utf-8"))
    except yaml.YAMLError:
        raise RuntimeError(
            "config.yaml in the app folder is not valid. Ask your administrator."
        ) from None
    SETTINGS = SETTINGS if isinstance(SETTINGS, dict) else {}
    PINNED = pinned_version(SETTINGS)
    LOCAL = SETTINGS.get("local_wheels", True)
    if not isinstance(LOCAL, bool):
        raise TypeError(
            "local_wheels in config.yaml must be true or false. Ask your administrator."
        )
    # local_wheels: a wheel in the app folder wins (e.g. a release candidate); else PyPI
    WHEEL = pick_wheel(APP, PINNED) if LOCAL else None
    if WHEEL is not None:
        install(WHEEL, "the app folder")
        print(f"Installed {WHEEL.name} from the app folder.")
    else:
        if LOCAL:
            print(
                f"No wheel {PINNED} in the app folder; installing from PyPI."
                if PINNED
                else "No wheel in the app folder; installing from PyPI."
            )
        install(f"{PACKAGE}=={PINNED}" if PINNED else PACKAGE, "PyPI")
        print(f"Installed {PACKAGE} {version(PACKAGE)} from PyPI.")

In [ ]:
# @title 3. Read your API keys
import os

from google.colab import userdata

from recording_transcriber import notebook

# one Colab secret per key variable of config.yaml, with the same name
NAMES = notebook.required_keys(APP / "config.yaml")
if NAMES is None:
    raise RuntimeError(
        "config.yaml in the app folder is not valid. Ask your administrator."
    )
MISSING = []
for NAME in NAMES:
    # GOOGLE_API_KEY is still accepted for the Gemini key
    for SECRET in (NAME, "GOOGLE_API_KEY") if NAME == "GEMINI_API_KEY" else (NAME,):
        try:
            VALUE = userdata.get(SECRET)
        except userdata.SecretNotFoundError:
            continue
        except userdata.NotebookAccessError:
            raise RuntimeError(
                f"Allow access to the secret {SECRET}: click the key icon on the left "
                "and switch on 'Notebook access', then Run all again."
            ) from None
        if VALUE:
            os.environ[NAME] = VALUE
            print(f"Using the secret {SECRET}.")
            break
    else:
        MISSING.append(NAME)
if MISSING:
    raise RuntimeError(
        f"Missing secret(s): {', '.join(MISSING)}. Add them with the key icon on the left "
        "(Notebook access on), then Run all again. See how-to.md, part 1."
    )

In [ ]:
# @title 4. Choose the job
from recording_transcriber import notebook
from recording_transcriber.cli import setup_logging

setup_logging()
JOBS = notebook.workspace_path(MY_DRIVE, WORKSPACE)
JOB_DIR = notebook.pick_job(JOBS, JOB) if JOBS is not None else None
if JOB_DIR is None:
    raise RuntimeError("No job to run: see the messages above.")
print(f"Job: {JOB_DIR.name}")

In [ ]:
# @title 5. Run
if notebook.run(JOB_DIR, APP / "config.yaml"):
    print(f"\nDone. Results in {JOB_DIR / 'out'}:")
    for path in sorted((JOB_DIR / "out").iterdir()):
        print(f"  {path.name}")
else:
    print(
        "\nThe run stopped. Choose Run all again to continue where it stopped. "
        "If it fails again, send your administrator the file work/run.log from the job folder."
    )